# AgentForge Kaggle MVP Demo

Run this notebook on Kaggle to produce a full portable-agent workflow:
- clone this repository
- install dependencies
- scan a sample workspace
- generate + validate manifest
- render preview
- export package artifact

Outputs are written under `/kaggle/working` for direct export/download.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORKING = Path('/kaggle/working')
REPO_URL = 'https://github.com/FateFumbler/agentforge.git'
REPO_DIR = WORKING / 'agentforge'
ARTIFACT_DIR = WORKING / 'artifacts'
SAMPLE_DIR = WORKING / 'sample'
SAMPLE_SRC = REPO_DIR / 'examples' / 'openclaw-sample'
SCAN_JSON = ARTIFACT_DIR / 'scan.json'
MANIFEST_JSON = ARTIFACT_DIR / 'manifest.json'
PREVIEW_JSON = ARTIFACT_DIR / 'preview.json'
PACKAGE_ZIP = WORKING / 'agentforge-package.zip'
IMPORT_DIR = WORKING / 'imported-workspace'


def run(cmd, **kwargs):
    pretty = ' '.join(str(x) for x in cmd)
    print(f'$ {pretty}')
    subprocess.run(cmd, check=True, **kwargs)


# 1) Clone repo if needed
if not REPO_DIR.exists():
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print(f'Repo already exists at {REPO_DIR}, skipping clone')

os.chdir(REPO_DIR)

# 2) Install runtime dependencies
run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'])
run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])

# 3) Prepare sample workspace + outputs
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
if SAMPLE_DIR.exists():
    shutil.rmtree(SAMPLE_DIR)
shutil.copytree(SAMPLE_SRC, SAMPLE_DIR)

# 4) Run full scan-to-package flow
run([sys.executable, '-m', 'agentforge.cli', 'scan', str(SAMPLE_DIR), '--output', str(SCAN_JSON)])
run([
    sys.executable,
    '-m',
    'agentforge.cli',
    'generate-manifest',
    '--scan',
    str(SCAN_JSON),
    '--output',
    str(MANIFEST_JSON),
])
run([sys.executable, '-m', 'agentforge.cli', 'validate-manifest', '--manifest', str(MANIFEST_JSON)])
run([sys.executable, '-m', 'agentforge.cli', 'preview', '--manifest', str(MANIFEST_JSON), '--output', str(PREVIEW_JSON)])
run([
    sys.executable,
    '-m',
    'agentforge.cli',
    'package',
    '--manifest',
    str(MANIFEST_JSON),
    '--workspace',
    str(SAMPLE_DIR),
    '--output',
    str(PACKAGE_ZIP),
])
run([sys.executable, '-m', 'agentforge.cli', 'import', '--artifact', str(PACKAGE_ZIP), '--output', str(IMPORT_DIR), '--overwrite'])

# 5) Quick output sanity check
for path in [SCAN_JSON, MANIFEST_JSON, PREVIEW_JSON, PACKAGE_ZIP]:
    assert path.exists(), f'Missing expected output: {path}'
    print(f'✓ {path} = {path.stat().st_size} bytes')
print('✓ Kaggle MVP pipeline complete')
